In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../agent/dataset/data/travelsense_india_trips.csv")

user_profiles = df.groupby("user_id").agg(
    total_trips=("trip_id", "count"),
    avg_budget=("total_cost_inr", "mean"),
    avg_duration=("trip_days", "mean"),
    avg_distance=("distance_km", "mean")
).reset_index()

features_to_scale = ["total_trips", "avg_budget", "avg_duration", "avg_distance"]

scaler = StandardScaler()
user_profiles_scaled = user_profiles.copy()
user_profiles_scaled[features_to_scale] = scaler.fit_transform(user_profiles[features_to_scale])

print("=== TASK 1: USER PROFILES CREATED & SCALED ===")
display(user_profiles.head())

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)

features = user_profiles_scaled[["total_trips", "avg_budget", "avg_duration", "avg_distance"]]

kmeans.fit(features)

user_profiles["cluster"] = kmeans.labels_

print("=== TASK 2: K-MEANS CLUSTERING COMPLETE ===")
display(user_profiles[["user_id", "total_trips", "avg_budget", "avg_duration", "cluster"]].head(10))

In [ ]:
cluster_analysis = user_profiles.groupby("cluster")[["total_trips", "avg_budget", "avg_duration"]].mean().round(2)

print("=== DETECTIVE WORK: ANALYZING THE 4 GROUPS ===")
display(cluster_analysis)

def assign_label(row):
    if row["avg_budget"] > 55000:
        return "Luxury Travelers"
    elif row["total_trips"] > 11:
        return "Frequent Backpackers"
    else:
        return "Standard Travelers"

user_profiles["traveler_segment"] = user_profiles.apply(assign_label, axis=1)

print("\n=== TASK 3: HUMAN LABELS ASSIGNED ===")
display(user_profiles[["user_id", "total_trips", "avg_budget", "traveler_segment"]].head())